# Experiment 1 — eval analysis

Operational review + Phase B planning for any promptfoo Exp 1 eval. Set
`EVAL_ID` below; everything else is read from `the released prediction CSVs`.
Spec: [`exp1_eval_analysis_design.md`](exp1_eval_analysis_design.md). Three-way
outcome (correct / wrong / format_failure) per PLAN.md §4.1; CI-overlap
finalist rule per PLAN.md §2.3. Designed to be re-runnable on smoke / pilot /
Phase A / Phase B / future evals — no lemma, provider, or template list is
hardcoded except the rate card and the canonical template-filename map (both
in [`_eval_loader.py`](_eval_loader.py)).

## Parameters

In [ ]:
# Pick one or more released prediction-table runs by name (see the `run` column in llm_predictions/*.csv).
# The registry maps short names to actual eval IDs + descriptions so this
# notebook (and the Phase B sibling) don't need to copy-paste IDs around.
# Loading multiple evals concatenates them and derives a yaml_kind column
# from the template — so plain + JSON Phase A runs become a joint analysis
# where output-format is a fourth configuration axis. Asymmetric axis
# coverage is handled per-section: the §4 yaml_kind axis test intersects
# to common cells; marginals use each eval's own coverage.
EVAL_NAMES = ['phase_a_plain_restricted', 'phase_a_json_restricted']
BOOTSTRAP_N = 10_000                  # PLAN.md §2.2 — bootstrap resamples
RNG_SEED = 20260516

In [ ]:
import datetime as _dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from _eval_loader import (load_evals, infer_stage, wald_ci, bootstrap_ci_binary,
                          mcnemar, holm_bonferroni, AXES, AXIS_VALUES, RATES,
                          PHASE_A_LEMMAS, EVALS, eval_id)

# Resolve registry names → eval IDs. Raises a clear error if any name is
# unregistered or has id: null.
EVAL_IDS = [eval_id(name) for name in EVAL_NAMES]
metas, df = load_evals(*EVAL_IDS)
RNG = np.random.default_rng(RNG_SEED)
multi_yaml = df['yaml_kind'].nunique() > 1
print(f'loaded {len(df):,} rows from {len(metas)} eval(s); '
      f'{df.provider_label.nunique()} providers, {df.task_type.nunique()} task types, '
      f'yaml_kinds={sorted(df.yaml_kind.unique())}')
for name in EVAL_NAMES:
    entry = EVALS[name]
    print(f'  {name:<28} → {entry["id"]}  ({entry["description"]})')

## §1. What this eval did

Single scope-summary section so the analyst doesn't have to guess what's in
the bag before reading further sections.

In [ ]:
def scope_summary(meta, df, label='eval'):
    ts = _dt.datetime.fromtimestamp(meta['created_at_ms'] / 1000).isoformat(timespec='seconds')
    lemmas = sorted(df.lemma.unique())
    quotes_per_lemma = df.groupby('lemma')['quote_idx'].nunique()
    # Only show axis values actually present in this eval's subset.
    axes_seen = {a: sorted(df[a].dropna().unique().tolist()) for a in AXES}
    print(f'=== {label} ===')
    print(f'  id              : {meta["id"]}')
    print(f'  description     : {meta["description"]}')
    print(f'  created         : {ts}')
    print(f'  rows            : {len(df):,}')
    print(f'  inferred stage  : {infer_stage(set(lemmas))}')
    print(f'  lemmas ({len(lemmas)})    : {lemmas}')
    print(f'  quotes/lemma    : n={len(quotes_per_lemma)}  min={quotes_per_lemma.min()}  '
          f'median={int(quotes_per_lemma.median())}  max={quotes_per_lemma.max()}')
    print(f'  providers ({df.provider_label.nunique()})  : {sorted(df.provider_label.unique())}')
    print(f'  task types ({df.task_type.nunique()}) : {sorted(df.task_type.unique())}')
    for a, vs in axes_seen.items():
        print(f'  axis {a:<14}: {vs}')
    print(f'  total spend     : ${df.cost.sum():.4f}')
    print(f'  outcomes        : {df.outcome.value_counts().to_dict()}')

# Per-eval summary (each eval is one row of EVAL_IDS / one slice of df by eval_id).
for m in metas:
    scope_summary(m, df[df.eval_id == m['id']], label=m['id'])
    print()

# Joint summary if multiple evals are loaded.
if len(metas) > 1:
    joint_meta = {'id': '+'.join(m['id'] for m in metas),
                  'description': 'JOINT (concatenated)',
                  'created_at_ms': max(m['created_at_ms'] for m in metas)}
    scope_summary(joint_meta, df, label='JOINT (analysis dataframe)')
    print(f'\nyaml_kind row counts: {df.yaml_kind.value_counts().to_dict()}')

## §2. Per-provider performance — the headline table

Three-way outcome separation per PLAN.md §4.1: accuracy is computed on the
full denominator (format failures count *against* the model, but are reported
separately so they can be diagnosed). 95% binomial CI is the normal approx
(`±1.96 √p(1-p)/n`). Reasoning tokens > 50 flagged as likely route quirk —
the Gemini reasoning-leak finding lives in [`docs/methodology_log.md`](../docs/methodology_log.md)
entry `g`. `$/correct` = cost per correct answer ([`exp1_cost_analysis.md`](exp1_cost_analysis.md)).

In [ ]:
def per_provider_table(df, by_yaml_kind=None):
    """Per-provider headline. If by_yaml_kind is None, auto-splits when the df
    contains more than one yaml_kind (so the plain/json asymmetric-denominator
    problem doesn't show up in a marginal average)."""
    if by_yaml_kind is None:
        by_yaml_kind = df['yaml_kind'].nunique() > 1
    group_cols = ['provider_label', 'yaml_kind'] if by_yaml_kind else ['provider_label']
    rows = []
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)
        n = len(g)
        ok = int(g.success.sum())
        ff = int((g.outcome == 'format_failure').sum())
        wr = int((g.outcome == 'wrong').sum())
        lo, hi = wald_ci(ok, n)
        avg_r = g.reasoning_tokens.mean()
        spent = g.cost.sum()
        row = {'provider' if c == 'provider_label' else c: k for c, k in zip(group_cols, keys)}
        row.update({
            'n': n,
            'acc%': round(100 * ok / n, 2),
            'ci_lo%': round(100 * lo, 2),
            'ci_hi%': round(100 * hi, 2),
            'fmt_fail%': round(100 * ff / n, 2),
            'wrong%': round(100 * wr / n, 2),
            'avg_r_tok': round(avg_r, 1),
            'spent$': round(spent, 4),
            '$/correct': round(spent / ok, 6) if ok else float('inf'),
        })
        rows.append(row)
    return pd.DataFrame(rows).sort_values('acc%', ascending=False).reset_index(drop=True)

headline = per_provider_table(df)
headline

In [ ]:
# Forest plot: per-provider accuracy + 95% CI. When yaml_kind is split, plain
# and json get distinct markers per provider so the format effect is visible.
fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(headline))))
if 'yaml_kind' in headline.columns:
    providers = headline['provider'].unique()
    yloc = {p: i for i, p in enumerate(providers)}
    for yk, marker, color, off in [('plain', 'o', 'C0', -0.18), ('json', 's', 'C1', 0.18)]:
        sub = headline[headline['yaml_kind'] == yk]
        if sub.empty: continue
        y = [yloc[p] + off for p in sub['provider']]
        ax.errorbar(sub['acc%'], y,
                    xerr=[sub['acc%'] - sub['ci_lo%'], sub['ci_hi%'] - sub['acc%']],
                    fmt=marker, color=color, capsize=3, label=yk)
    ax.set_yticks(list(yloc.values())); ax.set_yticklabels(list(yloc.keys()))
    ax.legend()
else:
    y = np.arange(len(headline))
    ax.errorbar(headline['acc%'], y,
                xerr=[headline['acc%'] - headline['ci_lo%'], headline['ci_hi%'] - headline['acc%']],
                fmt='o', capsize=3)
    ax.set_yticks(y); ax.set_yticklabels(headline['provider'])
ax.invert_yaxis()
ax.set_xlabel('Accuracy % (95% CI)')
ax.set_title(f"Per-provider accuracy — {' + '.join(m['id'][:18] for m in metas)}")
ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

# Defensive flags.
flagged_r = headline[headline['avg_r_tok'] > 50]
if not flagged_r.empty:
    print('FLAG: providers with avg reasoning_tokens > 50 (likely route quirk — see docs/methodology_log.md g):')
    print(flagged_r[[c for c in ['provider', 'yaml_kind', 'avg_r_tok'] if c in flagged_r.columns]].to_string(index=False))
flagged_ff = headline[headline['fmt_fail%'] > 5]
if not flagged_ff.empty:
    print('\nFLAG: providers with fmt_fail% > 5 (parser-bug or refusal pattern; investigate in §6):')
    print(flagged_ff[[c for c in ['provider', 'yaml_kind', 'fmt_fail%'] if c in flagged_ff.columns]].to_string(index=False))

## §3. Per-template effects

Marginal accuracy per task type. When the joint df contains multiple
`yaml_kind` values, the table breaks out task × yaml_kind — the plain-vs-JSON
comparison is no longer a separate paired test here, it's the yaml_kind axis
in §4.

Research context: Min et al. EMNLP 2022 predicts fewshot teaches format →
expect lower fmt_fail; Sumanathilaka et al. 2026 predicts rationale-rich
justified > zeroshot; Mishra et al. ACL 2022 supports decomposed.

In [ ]:
def per_template_table(df, by_yaml_kind=None):
    """Marginal per-task-type accuracy. Auto-splits by yaml_kind when both
    plain and json templates are in df."""
    if by_yaml_kind is None:
        by_yaml_kind = df['yaml_kind'].nunique() > 1
    group_cols = ['task_type', 'yaml_kind'] if by_yaml_kind else ['task_type']
    rows = []
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)
        n = len(g); ok = int(g.success.sum()); ff = int((g.outcome == 'format_failure').sum())
        lo, hi = wald_ci(ok, n)
        row = dict(zip(group_cols, keys))
        row.update({'n': n, 'acc%': round(100*ok/n, 2),
                    'ci_lo%': round(100*lo, 2), 'ci_hi%': round(100*hi, 2),
                    'fmt_fail%': round(100*ff/n, 2)})
        rows.append(row)
    return pd.DataFrame(rows).sort_values('acc%', ascending=False).reset_index(drop=True)

tpl_table = per_template_table(df)
tpl_table

**Sensitivity check — arbitrary filter: drop llama-3.2-3b only.** llama-3.2-3b
accounts for ~97 % of the eval's format failures, almost all on `justified`
(3,215 / 3,380 cells; see §6 below). Because format failures count as wrong in
`acc%`, llama-3.2-3b's behaviour disproportionately penalises `justified` in
the marginal table above. The table below repeats §3(a) **excluding that one
provider** — a one-off diagnostic; `df` itself is not modified, so §§4–6
continue to use the full data.

In [ ]:
per_template_table(df[df.provider_label != 'llama-3.2-3b-instruct'])

## §4. Axis effects (def_language, granularity, show_metadata, yaml_kind)

Each test case ideally appears at every axis-cell so McNemar pairs cleanly,
but when the joint df mixes a full-matrix plain eval with a restricted-matrix
json eval, the pairing key for the yaml_kind axis is restricted to the
**intersection** of cells (granularity, def_language, show_metadata) covered
by *both* yamls — typically (l1, *, False). Holm-Bonferroni across the (up to)
four axis tests.

Research grounding:
- `def_language` (la vs pl): Etxaniz et al. NAACL 2024 (English-pivot processing).
- `granularity` (l1 vs l1l2): Maru et al. ACL 2022 (coarse vs fine).
- `show_metadata`: no strong prior; PROMPT_RECOMMENDATIONS.md §3.4 sets a
  null-confirmation drop rule.
- `yaml_kind` (plain vs json): output-format/constrained-decoding effect — the
  PLAN.md §2.2 second axis the budget yamls were built around. JSON should
  reduce fmt_fail (free for free under constrained decoding) at the cost of
  any cognitive load from the schema.

In [ ]:
def _paired_subset(df, axis):
    """Return (sub_df, n_pairs) — rows restricted to cells where both values
    of `axis` are present. Used by both marginals and paired McNemar so they
    report apples-to-apples numbers under asymmetric eval coverage (e.g.
    plain full matrix + json restricted matrix)."""
    values_present = [v for v in AXIS_VALUES[axis] if v in df[axis].unique()]
    if len(values_present) < 2:
        return df.iloc[0:0], 0
    v0, v1 = values_present
    KEY = ['provider_label', 'lemma', 'quote_idx', 'task_type'] + [a for a in AXES if a != axis]
    a_idx = df[df[axis] == v0].set_index(KEY).index
    b_idx = df[df[axis] == v1].set_index(KEY).index
    common = a_idx.intersection(b_idx)
    if len(common) == 0:
        return df.iloc[0:0], 0
    keep = df.set_index(KEY).loc[common.unique()].reset_index()
    return keep[keep[axis].isin(values_present)], len(common)


def axis_marginals(df, axis):
    """Marginal accuracy per axis value, on the paired-cell intersection."""
    sub, _ = _paired_subset(df, axis)
    values_present = [v for v in AXIS_VALUES[axis] if v in sub[axis].unique()]
    rows = []
    for v in values_present:
        g = sub[sub[axis] == v]
        n = len(g); ok = int(g.success.sum())
        lo, hi = wald_ci(ok, n)
        rows.append({axis: str(v), 'n': n, 'acc%': round(100*ok/n,2),
                     'ci_lo%': round(100*lo,2), 'ci_hi%': round(100*hi,2)})
    return pd.DataFrame(rows)


def axis_paired_mcnemar(df, axis):
    """Paired McNemar between the two values of `axis`, on the paired-cell intersection."""
    sub, _ = _paired_subset(df, axis)
    values_present = [v for v in AXIS_VALUES[axis] if v in sub[axis].unique()]
    if len(values_present) < 2:
        return None
    v0, v1 = values_present
    KEY = ['provider_label', 'lemma', 'quote_idx', 'task_type'] + [a for a in AXES if a != axis]
    a = sub[sub[axis] == v0].set_index(KEY)['success']
    b = sub[sub[axis] == v1].set_index(KEY)['success']
    idx = a.index.intersection(b.index)
    if len(idx) == 0:
        return None
    a, b = a.loc[idx], b.loc[idx]
    b_only = int(((a == 1) & (b == 0)).sum())  # v0 correct, v1 wrong
    c_only = int(((a == 0) & (b == 1)).sum())  # v0 wrong, v1 correct
    chi2, p = mcnemar(b_only, c_only)
    return {'axis': axis, 'val_low': str(v0), 'val_high': str(v1),
            'n_paired': len(idx),
            'acc_low%': round(100*a.mean(),2), 'acc_high%': round(100*b.mean(),2),
            'delta_pp': round(100*(b.mean()-a.mean()),2),
            'mcnemar_p': p}


print('Marginals restricted to the paired-cell intersection per axis — they line up')
print('with acc_low%/acc_high% in the McNemar table below (necessary under asymmetric coverage).\n')
for axis in AXES:
    if df[axis].nunique() < 2:
        print(f'--- axis: {axis} (only one value present — skipped) ---\n')
        continue
    m = axis_marginals(df, axis)
    if m.empty:
        print(f'--- axis: {axis} (no paired cells — skipped) ---\n')
        continue
    print(f'--- axis: {axis} ---')
    print(m.to_string(index=False))
    print()

rows = [axis_paired_mcnemar(df, a) for a in AXES]
rows = [r for r in rows if r is not None]
axes_t = pd.DataFrame(rows)
axes_t['p_holm'] = holm_bonferroni(axes_t['mcnemar_p'].tolist())
print(f'Paired McNemar across axes (Holm-Bonferroni across the {len(axes_t)}-test family):')
axes_t

In [ ]:
# Bar charts: marginal accuracy per axis value, one panel per axis present.
present_axes = [a for a in AXES if df[a].nunique() >= 2]
fig, axs = plt.subplots(1, len(present_axes), figsize=(3.5 * len(present_axes), 3))
if len(present_axes) == 1:
    axs = [axs]
for ax, axis in zip(axs, present_axes):
    m = axis_marginals(df, axis)
    ax.bar(m[axis].astype(str), m['acc%'])
    err_lo = m['acc%'] - m['ci_lo%']; err_hi = m['ci_hi%'] - m['acc%']
    ax.errorbar(m[axis].astype(str), m['acc%'], yerr=[err_lo, err_hi], fmt='none', color='black', capsize=3)
    ax.set_title(axis); ax.set_ylabel('acc%')
    ax.set_ylim(0, max(m['ci_hi%'].max() * 1.1, 1))
plt.tight_layout(); plt.show()

## §5. Per-lemma × provider heatmap — cross-lemma stability

The generalisation claim the paper rests on is that the top-N providers are
*consistently* top across lemmas (PLAN.md §2.2). High Pearson correlation
between providers across lemmas = they agree which lemmas are hard. Low
std per provider across lemmas = stable performer.

In [ ]:
heat = (df.groupby(['provider_label', 'lemma'])['success'].mean() * 100).unstack()
heat = heat.reindex(index=heat.mean(axis=1).sort_values(ascending=False).index)
heat['MEAN'] = heat.mean(axis=1).round(2)
heat.loc['MEAN'] = heat.mean(axis=0).round(2)
heat.loc['STD']  = heat.iloc[:-1].std(axis=0).round(2)

fig, ax = plt.subplots(figsize=(max(8, 0.7 * heat.shape[1]), 0.45 * heat.shape[0] + 1.5))
data = heat.iloc[:-2, :-1]  # drop MEAN row/col + STD row for the heatmap colour scale
sns.heatmap(data, annot=True, fmt='.1f', cmap='YlGnBu', ax=ax,
            cbar_kws={'label': 'acc %'})
ax.set_title(f"Per-lemma × provider accuracy — {' + '.join(m['id'][:18] for m in metas)} (marginal across yaml_kind)")
plt.tight_layout(); plt.show()
heat

In [ ]:
# Pearson correlation between providers (each provider = vector of per-lemma accuracies).
vec = (df.groupby(['provider_label', 'lemma'])['success'].mean()).unstack()
corr = vec.T.corr().round(2)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, center=0, ax=ax)
ax.set_title('Provider × provider Pearson r (across-lemma accuracy vectors)')
plt.tight_layout(); plt.show()
print(f'mean off-diagonal r = {(corr.values[np.triu_indices_from(corr, k=1)]).mean():.2f}  '
      f'(high = providers agree which lemmas are hard)')

## §6. Failure analysis

Provider × template format-fail rate + top failure-reason families. Catches
route quirks (e.g. the justified-template fmt_fail observed in Phase A plain
— spec §6: 'investigate whether models emit a valid but unparseable variant
of "SENSE: N"') and per-provider regressions.

In [ ]:
# Format-failure rate per provider × task × yaml_kind. With JSON's constrained
# decoding the json yaml should have near-zero fmt_fail across the board; any
# non-zero cell on the json side is a route quirk worth investigating.
group_cols = ['provider_label', 'yaml_kind', 'task_type'] if multi_yaml else ['provider_label', 'template']
ff_rate = (df.assign(is_ff=(df.outcome == 'format_failure'))
             .groupby(group_cols)['is_ff']
             .mean() * 100).round(2)
if multi_yaml:
    # Wide format: rows = provider, columns = (yaml_kind, task_type) — easier to scan plain↔json per task.
    ff_rate = ff_rate.unstack(['yaml_kind', 'task_type']).fillna(0).round(2)
else:
    ff_rate = ff_rate.unstack().fillna(0).round(2)
print(f'Format-failure rate (%) per provider × {"yaml_kind × task" if multi_yaml else "template"}:')
ff_rate

In [ ]:
# Top 3 failure-reason families per (provider × template) cell, plus an output preview.
def reason_family(reason):
    if not isinstance(reason, str): return '(none)'
    if reason.startswith('format_failure'):
        return reason.split('|', 1)[0].strip()  # 'format_failure: no sense number in output'
    if reason.startswith('wrong'):
        return 'wrong: <expected/got>'
    return reason[:60]

for prov in sorted(df.provider_label.unique()):
    sub = df[(df.provider_label == prov) & (df.outcome == 'format_failure')]
    if sub.empty: continue
    print(f'\n--- {prov} — {len(sub)} format failures ---')
    families = sub.reason.map(reason_family).value_counts().head(3)
    for fam, c in families.items():
        print(f'  {c:>5}× {fam}')
    example = sub.iloc[0]
    print(f'  example output ({example.template}, {example.lemma} q{example.quote_idx}):')
    print(f'    {(example.output or "")[:200]!r}')

## §7. CI-overlap finalist selection — the Phase B promotion gate

Per PLAN.md §2.3:
1. A **configuration** is a (task_type × axis-cell) combination over the four
   axes (granularity × def_language × show_metadata × yaml_kind). With both
   plain (4 task × 8 plain-axis-cells = 32) and the restricted JSON (4 task
   × 2 json-axis-cells = 8) loaded, the joint config space is **40
   configurations** for §7; the §7b filter narrows it to the Phase-B axes.
2. For each (provider, configuration), compute accuracy + bootstrap CI
   (10K resamples — PLAN.md §2.2).
3. For each provider, find the top-accuracy configuration's CI.
4. Keep every configuration whose CI overlaps the top configuration's CI.
5. Union across providers → the paper-wide finalist set carried into
   Phase B (which runs on the disjoint Phase B lemma set).

Underlying rationale: Pezeshkpour & Hruschka NAACL 2024 (small-screening-set
winner's curse). PLAN_BUDGET.md §5.1 — CI-overlap operates on direct
evidence under the budget scenario.

In [ ]:
def config_id(row):
    return (f"{row.task_type}|{row.yaml_kind}|"
            f"gran={row.granularity}|def={row.def_language}|meta={int(bool(row.show_metadata))}")

df = df.assign(config=df.apply(config_id, axis=1))

rows = []
for (prov, cfg), g in df.groupby(['provider_label', 'config']):
    n = len(g); k = int(g.success.sum())
    lo, hi = bootstrap_ci_binary(k, n, n_boot=BOOTSTRAP_N, rng=RNG)
    rows.append({'provider': prov, 'config': cfg, 'n': n, 'acc': k/n,
                 'ci_lo': lo, 'ci_hi': hi})
config_perf = pd.DataFrame(rows)
n_configs = config_perf['config'].nunique()
# Expected: per yaml_kind, n_task × n_axis_cells_in_yaml.
expected = sum(df[df.yaml_kind == yk].groupby(['task_type','granularity','def_language','show_metadata']).ngroups
               for yk in df.yaml_kind.unique())
print(f'configurations evaluated: {n_configs}  (expected from yaml-by-yaml axis coverage: {expected})')

In [ ]:
# CI-overlap finalists: per provider, keep configs whose CI overlaps the top config's CI.
kept_per_provider = {}
diagnostics = []
for prov, g in config_perf.groupby('provider'):
    top = g.sort_values('acc', ascending=False).iloc[0]
    overlap_mask = (g['ci_hi'] >= top['ci_lo']) & (g['ci_lo'] <= top['ci_hi'])
    kept = g[overlap_mask]['config'].tolist()
    kept_per_provider[prov] = set(kept)
    diagnostics.append({'provider': prov, 'top_acc%': round(100*top['acc'],2),
                        'top_config': top['config'],
                        'top_ci%': f"[{round(100*top['ci_lo'],2)}, {round(100*top['ci_hi'],2)}]",
                        'n_kept': len(kept), 'n_total': len(g)})
diag = pd.DataFrame(diagnostics).sort_values('top_acc%', ascending=False)
print('Per-provider CI-overlap diagnostics:')
print(diag.to_string(index=False))

union = set().union(*kept_per_provider.values())
print(f'\nUnion across providers — paper-wide finalist set: {len(union)} of {n_configs} configurations')

# Spec §7 diagnostic: "If almost all → the rule isn't discriminating. If 1 → too strict."
if len(union) / n_configs >= 0.9:
    print(f'\nDIAGNOSTIC: union covers {len(union)/n_configs:.0%} of configs — rule is not discriminating.')
    print('  Common cause: a broken provider (top accuracy at or near zero) whose wide-relative CI overlaps every other config.')
    print('  Refinement applied in the next cell: drop providers whose top-CI upper bound is below a usability floor.')
elif len(union) <= 1:
    print(f'\nDIAGNOSTIC: union has {len(union)} config — rule is over-strict for this eval.')
    print('  Consider widening CI (lower bootstrap alpha) or relying on §3/§4 marginals + §6 diagnostics for the Phase B decision.')

In [ ]:
# Refinement: exclude broken providers from the union.
# A "broken provider" is one whose best configuration's CI doesn't separate from
# a usability floor (default 5%) — that provider can't discriminate between
# configurations, so its CI-overlap vote is uninformative and only inflates the
# union. PLAN.md §2.5 baseline reporting will note these providers separately.
USABILITY_FLOOR = 0.05  # top-config CI upper bound must exceed this to vote

broken = {prov for prov, g in config_perf.groupby('provider')
          if g['ci_hi'].max() < USABILITY_FLOOR}
healthy = {p for p in kept_per_provider if p not in broken}

if broken:
    print(f'Excluded broken providers (top-CI hi < {USABILITY_FLOOR:.0%}): {sorted(broken)}')
else:
    print(f'No broken providers under the {USABILITY_FLOOR:.0%} usability floor — refinement is a no-op for this eval.')

union_refined = set().union(*(kept_per_provider[p] for p in healthy)) if healthy else set()
print(f'Refined union (healthy providers only): {len(union_refined)} of {n_configs} configurations')

# Keep both objects available downstream; §9 surfaces the choice.
union_strict = set(union)
union = union_refined if (len(union) / n_configs >= 0.9 and broken) else union_strict
print(f'Using union of size {len(union)} for §8 cost projection and §9 keep/drop list.')

In [ ]:
# Per-config kept/dropped table.
config_summary = (config_perf
    .groupby('config')
    .agg(mean_acc=('acc','mean'), max_acc=('acc','max'), min_acc=('acc','min'))
    .assign(in_finalists=lambda d: d.index.isin(union),
            kept_by_n=lambda d: d.index.map(lambda c: sum(c in kept for kept in kept_per_provider.values())))
    .sort_values(['in_finalists','mean_acc'], ascending=[False, False]))
config_summary['mean_acc%'] = (100 * config_summary['mean_acc']).round(2)
config_summary['max_acc%']  = (100 * config_summary['max_acc']).round(2)
config_summary[['mean_acc%', 'max_acc%', 'kept_by_n', 'in_finalists']]

In [ ]:
# Heatmap: configurations × providers, kept (green) / dropped (grey) per provider.
kept_grid = pd.DataFrame(0, index=sorted(config_perf['config'].unique()),
                         columns=sorted(config_perf['provider'].unique()))
for prov, kept in kept_per_provider.items():
    for cfg in kept:
        kept_grid.loc[cfg, prov] = 1
# Annotate cells with each provider's accuracy on each config
acc_grid = config_perf.pivot(index='config', columns='provider', values='acc') * 100
fig, ax = plt.subplots(figsize=(max(8, 0.8 * kept_grid.shape[1]), 0.30 * kept_grid.shape[0] + 1.5))
sns.heatmap(acc_grid, annot=True, fmt='.0f', cmap='YlGnBu',
            mask=(kept_grid == 0), cbar_kws={'label': 'acc % (only kept-by-provider cells shown)'}, ax=ax)
ax.set_title('Configurations kept by CI-overlap rule, per provider (cells = acc%)')
plt.tight_layout(); plt.show()

In [ ]:
print(f'=== Phase B promotion set ({len(union)}/{n_configs} configurations) ===')
for cfg in sorted(union):
    n_kept_by = sum(cfg in kept for kept in kept_per_provider.values())
    print(f'  [{n_kept_by:>2}/{len(kept_per_provider)} providers] {cfg}')
print(f'\nWe promote {len(union)} of {n_configs} configurations to Phase B.')

### §7b. Phase-B-axes view — arbitrary filter applied below

The §7 output above operates on the **joint plain+JSON config space**. Phase B
drops two axes a priori (per methodology-log entry `2026-05-16a`):

- `show_metadata=True` — the metadata block carried no model-actionable
  signal in Phase A.
- `granularity=l1l2` — two-level granularity deferred to a future study.

Both `def_language` values (la, pl) and both `yaml_kind` values (plain, json)
are retained. The cell below **arbitrarily reassigns `df` in place** to the
kept axis subset (`granularity=l1`, `show_metadata=False`) and reruns the
CI-overlap finalist gate. With both yamls loaded the surviving config count
is **4 templates × 2 def_languages × 2 yaml_kinds = 16**; with only one yaml,
it's 8. All cells from here onward (§8 cost, §9 recommendations) operate on
the filtered subset — so the cost projection reflects the Phase B run scope.

In [ ]:
# ARBITRARY FILTER — apply the Phase B axis cut to `df` in place and rerun §7.
# After this cell, all sections below operate on the filtered subset
# (granularity=l1, show_metadata=False). The dropped axes' data still
# exists in the underlying evals on disk; we just don't analyse them past
# this point. Both yaml_kinds (plain, json) are retained.
df = df[(df.granularity == 'l1') & (df.show_metadata == False)].copy()
print(f'rows after Phase B axis filter: {len(df):,} '
      f'(yaml_kind breakdown: {df.yaml_kind.value_counts().to_dict()})\n')

# Recompute per-config performance + bootstrap CI on the filtered subset.
rows = []
for (prov, cfg), g in df.groupby(['provider_label', 'config']):
    n = len(g); k = int(g.success.sum())
    lo, hi = bootstrap_ci_binary(k, n, n_boot=BOOTSTRAP_N, rng=RNG)
    rows.append({'provider': prov, 'config': cfg, 'n': n, 'acc': k/n,
                 'ci_lo': lo, 'ci_hi': hi})
config_perf = pd.DataFrame(rows)
n_configs = config_perf['config'].nunique()
expected_filtered = 4 * df.def_language.nunique() * df.yaml_kind.nunique()
print(f'configurations evaluated (filtered): {n_configs} '
      f'(expected: 4 templates × {df.def_language.nunique()} def_languages '
      f'× {df.yaml_kind.nunique()} yaml_kinds = {expected_filtered})\n')

# Re-apply CI-overlap promotion rule on the filtered configs (cell-72 logic).
kept_per_provider = {}
for prov, g in config_perf.groupby('provider'):
    top = g.sort_values('acc', ascending=False).iloc[0]
    overlap_mask = (g['ci_hi'] >= top['ci_lo']) & (g['ci_lo'] <= top['ci_hi'])
    kept_per_provider[prov] = set(g[overlap_mask]['config'])

# Re-apply broken-provider refinement (cell-684f08df logic).
broken = {prov for prov, g in config_perf.groupby('provider')
          if g['ci_hi'].max() < USABILITY_FLOOR}
healthy = {p for p in kept_per_provider if p not in broken}
if broken:
    print(f'Excluded broken providers (top-CI hi < {USABILITY_FLOOR:.0%}): {sorted(broken)}')
union_refined = set().union(*(kept_per_provider[p] for p in healthy)) if healthy else set()
union_strict  = set().union(*kept_per_provider.values())
union = union_refined if (len(union_strict) / n_configs >= 0.9 and broken) else union_strict
print(f'Refined union: {len(union)} of {n_configs} configurations\n')

print(f'=== Phase-B-axes promotion set ({len(union)}/{n_configs} configurations) ===')
for cfg in sorted(union):
    n_kept_by = sum(cfg in kept for kept in kept_per_provider.values())
    print(f'  [{n_kept_by:>2}/{len(kept_per_provider)} providers] {cfg}')

## §8. Cost summary

Total spend by provider, cost per correct answer, and a Phase B projection
based on the §7 finalist set.

In [ ]:
cost_table = (df.groupby('provider_label')
                .agg(n=('success','size'), ok=('success','sum'), spent=('cost','sum'),
                     ff=('outcome', lambda s: int((s=='format_failure').sum())))
                .assign(acc_pct=lambda d: round(100*d.ok/d.n, 2),
                        per_correct=lambda d: round(d.spent / d.ok, 6))
                .sort_values('per_correct'))
cost_table.loc['TOTAL'] = [cost_table['n'].sum(), cost_table['ok'].sum(), cost_table['spent'].sum(),
                          cost_table['ff'].sum(),
                          round(100*cost_table['ok'].sum()/cost_table['n'].sum(), 2),
                          round(cost_table['spent'].sum()/cost_table['ok'].sum(), 6)]
cost_table

In [ ]:
# Phase B cost projection: (Phase A spend) × (Phase B lemmas / Phase A lemmas) × (kept configs / total configs)
# Phase B lemmas = lemmas with >= 10 quotes excluding Phase A — would need exp1_tests.py to enumerate. Approx: same as Phase A.
n_lemmas_A = df.lemma.nunique()
frac_configs = len(union) / n_configs
phaseA_spend = df.cost.sum()
for nB in [n_lemmas_A, max(1, n_lemmas_A // 2), 2 * n_lemmas_A]:
    proj = phaseA_spend * (nB / n_lemmas_A) * frac_configs
    print(f'  Phase B with {nB} lemmas × {len(union)}/{n_configs} configs: ${proj:.4f}')
print(f'\n(read actual Phase B lemma count from scripts/exp1_tests.py:_phase_b_lemmas before locking the projection.)')

## §9. Pre-Phase B recommendations

Three subsections: keep / drop / open questions. The contents here are
*templated* — the analyst should edit them based on the specific findings
above. Don't auto-overwrite; this is a judgement-call cell.

In [ ]:
print('### Configurations to keep (from §7 CI-overlap rule):\n')
for cfg in sorted(union):
    print(f'  - {cfg}')

print('\n### Configurations to drop:\n')
dropped = sorted(set(config_perf['config'].unique()) - union)
for cfg in dropped:
    m = config_perf[config_perf['config'] == cfg]['acc'].mean()
    print(f'  - {cfg}  (mean acc across providers: {100*m:.1f}%)')

print('\n### Open questions for the writeup (edit this list based on §1–§6 findings):\n')
# Surface auto-detected concerns to seed the analyst's review.
weak = headline[headline['acc%'] < 5]
for _, r in weak.iterrows():
    print(f'  - {r["provider"]} at {r["acc%"]}% accuracy with {r["fmt_fail%"]}% format-failure — keep as floor reference or drop from Phase B?')
broken = headline[headline['fmt_fail%'] > 5]
for _, r in broken.iterrows():
    print(f'  - {r["provider"]} format-failure rate is {r["fmt_fail%"]}% — root-cause in §6 before Phase B.')
if not (weak.empty and broken.empty):
    print('  - (auto-surfaced; refine before writeup)')

## §10. How to read this notebook — statistical orientation

A reference cell so future-you (or a co-author) can interpret the tables
above without re-deriving the basis from the spec. Keep edits surgical;
this is not where new analysis goes.

### Holm-Bonferroni (§3, §4)

Running multiple hypothesis tests inflates the false-positive rate: at
α=0.05 per test, the probability of at least one false positive across 4
independent tests is ~1−(0.95)⁴ ≈ 19 %, not 5 %. Holm-Bonferroni sorts
p-values smallest-to-largest, multiplies the k-th by (m−k+1), and reports
the running maximum. After this adjustment, the **family-wise** error
rate stays at α=0.05 — meaning across the *family* of tests jointly, you
have <5 % chance of any false positive. It is strictly more powerful
than plain Bonferroni (which multiplies every p by m), and almost always
preserves more findings.

In §4 the family is the (up to) 4 axis tests. In a hypothetical §3(b)
with multiple template pairs the family is those pairs. The family
boundary is a methodological choice; pick it before looking at the data
and don't redraw it later.

### Statistical vs practical significance (§2, §4, §7)

With our sample sizes (n in the tens of thousands per axis cell), the
standard error on accuracy is sub-percentage-point: roughly
`±1.96·√(p·(1−p)/n)`. At n=67,600, p≈0.32 → ±0.35 pp. So almost any true
difference becomes statistically detectable. A p-value tells you "this
isn't zero," not "this matters."

The practical question is whether an effect crosses an action threshold.
For sense assignment in a lexicographer-facing pipeline, a 0.4 pp gain
is ~1 extra correct sense per 230 quotes — operationally invisible. A
5 pp gain is ~1 per 20, noticeable. A 17 pp gain (the plain↔json
effect on this run) is ~1 per 6 — dominant.

Rule of thumb for accepting or dropping an axis:

| Δpp | Holm p | reading |
|---:|---:|---|
| < 0.5 | any | operationally null; drop the axis |
| 1–3 | < 0.01 | real but small; keep only if directly paper-relevant |
| 3–10 | < 0.01 | substantive; should drive a config decision |
| > 10 | < 0.001 | dominant effect; structures the analysis |

### Confidence intervals (§2, §3, §4, §7)

A CI quantifies how much the point estimate could shift if you re-ran on
different data. We use them in three roles:

**Role A — separability ranking (§2, §3, §4).** Non-overlapping CIs let
you confidently order two estimates; overlapping CIs mean the order
might flip on resampling.

**Role B — small-screening-set winner's curse (§7).** This is the
load-bearing role. Per Pezeshkpour & Hruschka NAACL 2024: if config A
"wins" Phase A by 2 pp but the CIs of A and B overlap, the apparent
winner can flip on Phase B. The CI-overlap rule keeps both as Phase B
finalists rather than committing to A. The rule is conservative on
purpose — it costs Phase B compute but avoids wasting a run on a
spurious winner.

**Role C — projection bounds (§8).** Phase B cost projections are
themselves point estimates. Per-config CIs (from §7) tell you how much
Phase B per-config accuracy could differ from the Phase A point
estimate.

**Wald vs bootstrap.** §2/§3/§4 use the Wald normal-approximation CI
(`±1.96·√p(1−p)/n`) — fast and fine at our sample sizes. §7 uses
non-parametric bootstrap per PLAN.md §2.2; for a binary outcome with k
ones in n the resample distribution is Binomial(n, k/n) in closed form,
so the bootstrap is one numpy call per config and agrees with Wald to
within ~0.1 pp at our sizes. The choice is methodological consistency
with the registered plan, not numeric accuracy.

**Why CI width depends on n.** Halving n widens CI by √2. Per-provider
CIs (n=13,520) are ~±0.8 pp wide; per-config CIs (n≈422) are ~±4.7 pp
wide. That's why §7 looks "less discriminating" than §2 — with less data
per cell, wider CIs overlap more easily. This is honest, not a bug.

### How these signals combine for Phase B selection

The final selection (recorded in [`../docs/methodology_log.md`](../docs/methodology_log.md)
entry `2026-05-16c`) used four signals in this order:

1. **§4 Holm-adjusted p × Δpp** — settle the axes where the effect is
   either too large to need re-confirmation (plain↔json) or too small to
   matter (show_metadata).
2. **§2 per-provider direction uniformity** — if an axis effect goes the
   same way for ≥ 80 % of healthy providers, it's settled regardless of
   sample-size-driven CI overlap.
3. **§7b kept_by_n threshold** — among configs surviving (1) and (2),
   keep those with ≥ ~70 % healthy-provider consensus. This is where
   the CI-overlap rule's discrimination actually lives once you remove
   the broken-provider noise.
4. **§8 cost ceiling** — sanity check the resulting list against budget.

The CI-overlap union in §7 / §7b alone is a *filter*, not a decision:
it tells you which configs *could* win, not which ones the paper should
pay to test on Phase B lemmas. The decision rule above operationalises
the "pay to test" judgement.